# HAGRID Output Analysis – Usage Example

This notebook shows how to use the `hagrid_output_analysis` package to load,
parse, and analyse MATSim parcel‐delivery simulation outputs.

## 1 · Setup & Imports

In [ ]:
# Standard scientific stack
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

# Import the package
from hagrid_output_analysis import (
    ReferenceData,
    process_single_run,
    discover_runs,
    run_batch,
    config,
)

print("hagrid_output_analysis loaded successfully.")

## 2 · Load Reference Data

The `ReferenceData` class loads all shared datasets once:
- MATSim network (with road‑type classification)
- Region clusters (Raumtypen)
- Network → Raumtyp mapping
- PLZ area polygons

Adjust the paths below to match your workspace.

In [ ]:
ref = ReferenceData(
    regionclusters_path="input/regionclusters.pkl",
    network_path="input/simRes/BC_500_it_no_reduction/basecase_13052025.output_network.xml.gz",
    networkplus_path="input/networkplus.pkl",
    plz_areas_path="input/plz_areas.csv",
)

print(f"Network links:  {len(ref.network):,}")
print(f"Region clusters: {len(ref.regionclusters)}")
print(f"PLZ areas:       {len(ref.gdf_areas)}")

## 3 · Process a Single Run

Use `process_single_run()` to execute the full pipeline
(event parsing → vehicle stats → carrier demand → EV model → emissions)
for one simulation output.

In [ ]:
EVENT_FILE   = "input/simRes/batch/my_scenario/run_01/run_01.output_events.xml.gz"
CARRIER_FILE = "input/simRes/batch/my_scenario/run_01/run_01.output_carriers.xml.gz"

results = process_single_run(
    ref,
    event_file=EVENT_FILE,
    carrier_file=CARRIER_FILE,
    run_label="my_scenario/run_01",
)

### 3a · Inspect the enriched result DataFrame

In [ ]:
result_df = results["emissions_result"]
print(f"Vehicles: {len(result_df)}")
print(f"Columns:  {list(result_df.columns)}")
result_df.head()

### 3b · Quick emission summary

In [ ]:
long_df = results["emissions_15min_long"]

total_kg = long_df["emissions_g"].sum() / 1000
print(f"Total network emissions: {total_kg:,.1f} kg CO₂")

# Emissions by area type
by_area = (
    long_df.groupby("area_type")["emissions_g"]
    .sum()
    .div(1000)
    .sort_values(ascending=False)
)
by_area.plot.bar(title="Emissions by area type (kg CO₂)")
plt.tight_layout()
plt.show()

## 4 · Batch Processing

To process **all** simulation runs under a directory tree use
`discover_runs()` and `run_batch()`.

In [ ]:
BATCH_DIR  = r"input\simRes\batch"
OUTPUT_DIR = r"input\simRes\batch\processed"

runs = discover_runs(BATCH_DIR)
print(f"Found {len(runs)} runs:")
for r in runs:
    print(f"  {r['scenario']}/{r['run_name']}")

In [ ]:
# Uncomment to actually run the batch:
# run_batch(ref, runs=runs, output_dir=OUTPUT_DIR)

## 5 · Override Config at Runtime

All constants live in `hagrid_output_analysis.config`.
Override them **before** calling the pipeline functions.

In [ ]:
# Example: switch to CO2e basis with WTW
import hagrid_output_analysis.config as cfg

cfg.EMISSIONS_BASIS = "CO2e"
cfg.USE_WTW = True
cfg.GLOBAL_EV_TARGET = 0.30  # 30 % EV target

print(f"Basis:     {cfg.EMISSIONS_BASIS}")
print(f"WTW:       {cfg.USE_WTW}")
print(f"EV target: {cfg.GLOBAL_EV_TARGET:.0%}")

## 6 · Available Result Keys

The dict returned by `process_single_run()` contains the following keys:

In [ ]:
# (assuming 'results' is available from section 3)
for key in sorted(results.keys()):
    obj = results[key]
    if isinstance(obj, pd.DataFrame):
        desc = f"DataFrame {obj.shape}"
    elif isinstance(obj, dict):
        desc = f"dict ({len(obj)} entries)"
    else:
        desc = type(obj).__name__
    print(f"  {key:<35s} → {desc}")